Martpy 101
==========

This notebook is a simple a demo of basic functions of martpy.

In [1]:
# import package
import pandas as pd
import martpy as mart
from IPython.display import display

In [2]:
path_data = '../tests/data/mart_hallue'
# load a RMA project
rma = mart.read_rma_file(f'{path_data}/hallue.rma')
print(rma.get_files())

{'permh': 'hallue.permh', 'debit': 'hallue.debit', 'charg': 'hallue.charg', 'emmca': 0.001, 'emmli': 'hallue.emmli', 'zgeom': 'hallue.zgeom', 'hsubs': 'hallue.hsubs', 'topog': 'hallue.topog', 'zonep': 'hallue.zonep', 'idebo': 'hallue.idebo', 'mart': 'hallue.mart', 'pastp': 'hallue.pastp', 'aff_r': 'hallue.aff_r', 'trc_r': 'hallue.trc_r', 'lon_r': 'hallue.lon_r', 'lar_r': 'hallue.lar_r', 'hau_r': 'hallue.hau_r', 'epai_r': 0.5, 'perm_r': 'hallue.perm_r', 'qext_r': 'hallue.qext_r', 'flo_r': 'hallue.flo_r', 'rug_r': 0.02, 'pnt_r': 0.005, 'arb_r': 'hallue.arb_r', 'fon_r': 'hallue.fon_r', 'layer': 'hallue.layer', 'histo': 'hallue.histo', 'meteo': 'hallue.meteo', 'd_ava': 'hallue.d_ava', 'm_plu': 'hallue.m_plu', 'm_etp': 'hallue.m_etp', 'nzcli': 'hallue.nzcli', 'zoneg2': 'hallue.zoneg2', 'zoneg3': 'hallue.zoneg3'}


In [3]:
# load mart general parameters
param = mart.read_general_parameters(f'{path_data}/hallue.mart')
print(param.get_param_groups())  # list of all group of parameters
print(param['outputs'])  # print one group of parameters

['outputs', 'timesteps', 'solver', 'units', 'geometry_process', 'transport', 'density', 'thermal', 'trajectory_concentr', 'unsat_multiphas', 'gaz_oil_salt', 'oil_naq_flow', 'gaz_flow', 'geochemistry', 'optim', 'river', 'climatology', 'stdout', 'log_level', 'agriculture', 'init_parameters']
{'gen_list': 0, 'grid_list': 0, 'restart': 0, 'check_converg': 0, 'lig_converg': 0, 'col_converg': 0, 'lay_converg': 0, 'grid_converg': 0, 'n_iter_converg': 0, 'log_level': 0, 'check_only': 0, 'save_zone_balance': 0, 'save_gw_balance': 1, 'save_variations': 0, 'del_flow_balance': 0, 'time_serie_fmt': 0, 'save_outputs_every_step': 0, 'save_climatic_water_balance': 1, 'save_mass_therm_balance': 0, 'save_flow_balance_zone': 1, 'save_drain_balance': 0, 'save_chem_converg': 0, 'grid_fmt': 0, 'date_fmt': 0, 'velocity_xy_factor': 0, 'time_serie_frequency': 0, 'save_nitrates_balance': 0, 'write_xy_domain': 0, 'list_river_network': 0, 'compute_mass_therm_balance': 0}


In [7]:
# load timesteps
pastp = mart.read_mart_timesteps(f'{path_data}/hallue.pastp')
print(pastp)


Marthe timesteps
Datetime From: 1995-07-31 00:00:00, to: 2012-07-31 00:00:00
Number of steps: 6211
Number of steps with action: 1



In [4]:
# load histo
histo = mart.read_histo_file(f'{path_data}/hallue.histo')
display(histo)

,id_grid,variable,mode,layer,obs,name,x,y,interp,col,row
0,0,Charge,XCOO,1,1,00464X0013/H1,611.62,2560.0,True,<NA>,<NA>
1,0,Charge,XCOO,1,<NA>,00471X0010/H1,617.43,2558.7,True,<NA>,<NA>
2,1,Charge,COOR,1,<NA>,00471X0010/no_interp,617.43,2558.7,False,<NA>,<NA>
3,0,Débit_Rivi,MAIL,1,<NA>,E6397010,NaN,NaN,False,25,29
4,0,Débit_Rivi,MAIL,1,<NA>,NaN,NaN,NaN,False,22,40


In [5]:
# useful func to fill coordinates from grid
# grid = mart.grid.load_marthe_grid(fpermh, ...)
# histo.fill_coordinates_from_grid(grid)  # grid is a grid dataset. See below

## Edit timesteps

In [8]:
# from existing
pastp.add_times('2012-08-01', '2020-07-31')
pastp.add_action(steps=-1, variable='CALCUL_HDYNAM', action='ACTION', value=2)
print(pastp)
pastp.get_actions()['2020-07-31']


Marthe timesteps
Datetime From: 1995-07-31 00:00:00, to: 2020-07-31 00:00:00
Number of steps: 9133
Number of steps with action: 2



{'CALCUL_HDYNAM': [{'mode': 'ACTION', 'value': 2}]}

In [9]:
# from scratch
dt1 = pd.date_range('1990-01-01', '2000-01-01', freq='YS')
dt2 = pd.date_range('2000-02-01', '2025-07-31', freq='D')
custom_dt = dt1.union(dt2)
pastp2 = mart.MartheTimeSteps.from_dates(dates=custom_dt)
pastp2.set_met_forcings(['PLUIE_{}_{}', 'ETP_{}_{}'])  # wrapper method to set annual forcings
print(pastp2.get_actions())

{'2000-08-01': {'METEO_PLUIE': [{'mode': 'FICHIER', 'file': PosixPath('PLUIE_2000_2001')}], 'METEO_ETP': [{'mode': 'FICHIER', 'file': PosixPath('ETP_2000_2001')}], 'FLUX_PLUV': [{'mode': 'FICH_METE', 'step': '*'}], 'FLUX_ETP': [{'mode': 'FICH_METE', 'step': '*'}]}, '2001-08-01': {'METEO_PLUIE': [{'mode': 'FICHIER', 'file': PosixPath('PLUIE_2001_2002')}], 'METEO_ETP': [{'mode': 'FICHIER', 'file': PosixPath('ETP_2001_2002')}], 'FLUX_PLUV': [{'mode': 'FICH_METE', 'step': '*'}], 'FLUX_ETP': [{'mode': 'FICH_METE', 'step': '*'}]}, '2002-08-01': {'METEO_PLUIE': [{'mode': 'FICHIER', 'file': PosixPath('PLUIE_2002_2003')}], 'METEO_ETP': [{'mode': 'FICHIER', 'file': PosixPath('ETP_2002_2003')}], 'FLUX_PLUV': [{'mode': 'FICH_METE', 'step': '*'}], 'FLUX_ETP': [{'mode': 'FICH_METE', 'step': '*'}]}, '2003-08-01': {'METEO_PLUIE': [{'mode': 'FICHIER', 'file': PosixPath('PLUIE_2003_2004')}], 'METEO_ETP': [{'mode': 'FICHIER', 'file': PosixPath('ETP_2003_2004')}], 'FLUX_PLUV': [{'mode': 'FICH_METE', 'step

## Grids

martpy include a `gridmarthe` wrapper/accessor.

In [10]:
fpermh = '../tests/data/mart_hallue/hallue.permh'
ds = mart.grid.load_marthe_grid(fpermh, drop_nan=True, xyfactor=1e3)
print(ds)


<xarray.Dataset> Size: 30kB
Dimensions:  (time: 1, zone: 927)
Coordinates:
  * time     (time) datetime64[ns] 8B 1850-01-01
  * zone     (zone) int32 4kB 255 256 257 258 259 ... 2722 2723 2724 2725 2726
Data variables:
    permeab  (time, zone) float64 7kB 0.00062 0.00064 ... 0.001919 0.003067
    x        (zone) float32 4kB 6.178e+05 6.182e+05 ... 6.072e+05 6.078e+05
    y        (zone) float32 4kB 2.567e+06 2.567e+06 ... 2.543e+06 2.543e+06
    dx       (zone) float32 4kB 500.0 500.0 500.0 500.0 ... 500.0 500.0 500.0
    dy       (zone) float32 4kB 500.0 500.0 500.0 500.0 ... 500.0 500.0 500.0
    izone    (zone) int32 4kB 1 2 3 4 5 6 7 8 ... 921 922 923 924 925 926 927
Attributes: (12/17)
    conventions:          CF-1.10
    title:                
    marthe_grid_version:  9.0
    original_dimensions:  x,y,z [grids]: 53 54 1
    lon_resolution:       500.0
    lat_resolution:       500.0
    ...                   ...
    creation_date:        Created on 2025-12-17T14:03:36Z UTC
   

## Outputs

martpy also provide unified functions to read marthe outputs (time series of simulated variables
or water balance - grids outputs can be read as any marthe grid file with the `gridmarthe` module, see above).

In [12]:
df = mart.read_output_time_series(f'{path_data}/historiq.prn')
display(df.head())

field              Charge               Débit_Rivi                       
coords            X31_Y18       X42_Y21    X25_Y29   X22_Y40      Unknown
riv_afluent             @             @          @         @      AF1_TR1
name        00464X0013/H1 00471X0010/H1   E6397010         @ riv_par_affl
date                                                                     
1995-07-31       67.93486      74.64585   0.866505  1.685418       9999.0
1995-08-01       67.93485      74.64585   0.866505  1.685408       9999.0
1995-08-02       67.93485      74.64584   0.866505  1.685386       9999.0
1995-08-03       67.93485      74.64582   0.866505  1.685354       9999.0
1995-08-04       67.93484      74.64578   0.866505  1.685316       9999.0

In [13]:
df2 = mart.read_output_water_balance(f'{path_data}/histobil_debit.prn')
display(df2.head())

Entr_Limit  Sort_Limit  Entr_Inject_fix  Sort_Pompag_fix  \
zone   date                                                                   
global 1995-07-31    192385.4   -10941.12              0.0           0.0000   
       1995-08-01    192385.4   -10941.12              0.0        -625.1014   
       1995-08-02    192385.5   -10941.12              0.0        -625.1014   
       1995-08-03    192385.6   -10941.12              0.0        -625.1014   
       1995-08-04    192385.9   -10941.11              0.0        -625.1014   

                   Réduct_Pompage  Désaturation  Débord/Suintem  \
zone   date                                                       
global 1995-07-31             0.0           0.0       -79599.67   
       1995-08-01             0.0           0.0       -79600.01   
       1995-08-02             0.0           0.0       -79599.95   
       1995-08-03             0.0           0.0       -79599.91   
       1995-08-04             0.0           0.0       -79599.86   

                   Recharge_Maill  Évaporat_Maill  Excès_Irrigat  ...  \
zone   date                                                       ...   
global 1995-07-31    0.000000e+00             0.0            0.0  ...   
       1995-08-01    2.432245e-10             0.0            0.0  ...   
       1995-08-02    1.162146e-09             0.0            0.0  ...   
       1995-08-03    7.596600e-11             0.0            0.0  ...   
       1995-08-04    2.252325e-10             0.0            0.0  ...   

                   Déb._de_Transit  Glob_Lacs-Nappe  Glob_Éch_Phase  \
zone   date                                                           
global 1995-07-31              NaN              NaN             NaN   
       1995-08-01              NaN              NaN             NaN   
       1995-08-02              NaN              NaN             NaN   
       1995-08-03              NaN              NaN             NaN   
       1995-08-04              NaN              NaN             NaN   

                   Somme=Bil_Glob.  Entr_Lim_Z_Hau  Sort_Lim_Z_Hau  \
zone   date                                                          
global 1995-07-31              NaN             NaN             NaN   
       1995-08-01              NaN             NaN             NaN   
       1995-08-02              NaN             NaN             NaN   
       1995-08-03              NaN             NaN             NaN   
       1995-08-04              NaN             NaN             NaN   

                   Glob_Lim_Z_Hau  Entr_Lim_Z_Bas  Sort_Lim_Z_Bas  \
zone   date                                                         
global 1995-07-31             NaN             NaN             NaN   
       1995-08-01             NaN             NaN             NaN   
       1995-08-02             NaN             NaN             NaN   
       1995-08-03             NaN             NaN             NaN   
       1995-08-04             NaN             NaN             NaN   

                   Glob_Lim_Z_Bas  
zone   date                        
global 1995-07-31             NaN  
       1995-08-01             NaN  
       1995-08-02             NaN  
       1995-08-03             NaN  
       1995-08-04             NaN  

[5 rows x 58 columns]

by default, water balance are multiindex `pandas.DataFrame`. It is useful if water balance
is edited by zone. If you do not need it, you can read the global balance only and get rid
of the multiindex:

In [14]:
df2 = mart.read_output_water_balance(f'{path_data}/histobil_debit.prn', add_multiindex=False)
display(df2.head())

,Entr_Limit,Sort_Limit,Entr_Inject_fix,Sort_Pompag_fix,Réduct_Pompage,Désaturation,Débord/Suintem,Recharge_Maill,Évaporat_Maill,Excès_Irrigat,...,d_BilQ<->QFixé,Glob_Drain_Napp,Glob_Lacs_Nappe,d_BilanVol_Pas,%_Non_CV_Gl_Pas,Nombre_Désatur,Nombre_Suintem,Nombre_Itérat,Nombre_Sous-pas,Nbre_Réduc_Pomp
date,,,,,,,,,,,,,,,,,,,,,
1995-07-31,192385.4,-10941.12,0.0,0.0000,0.0,0.0,-79599.67,0.000000e+00,0.0,0.0,...,-0.050260,0.0,0.0,-0.050260,2.304042e-05,2.0,7.0,50.0,1.0,0.0
1995-08-01,192385.4,-10941.12,0.0,-625.1014,0.0,0.0,-79600.01,2.432245e-10,0.0,0.0,...,-0.002837,0.0,0.0,-0.002835,1.300660e-06,2.0,7.0,14.0,1.0,0.0
1995-08-02,192385.5,-10941.12,0.0,-625.1014,0.0,0.0,-79599.95,1.162146e-09,0.0,0.0,...,-0.005776,0.0,0.0,-0.005774,2.649201e-06,2.0,7.0,13.0,1.0,0.0
1995-08-03,192385.6,-10941.12,0.0,-625.1014,0.0,0.0,-79599.91,7.596600e-11,0.0,0.0,...,-0.005520,0.0,0.0,-0.005518,2.532001e-06,2.0,7.0,13.0,1.0,0.0
1995-08-04,192385.9,-10941.11,0.0,-625.1014,0.0,0.0,-79599.86,2.252325e-10,0.0,0.0,...,-0.001170,0.0,0.0,-0.001169,5.365525e-07,2.0,7.0,15.0,1.0,0.0
